In [1]:
import pandas as pd

path = r"C:\Users\91766\Downloads\airline_booking_dataset\3_seats.csv"
df = pd.read_csv(path, dtype = str)

In [2]:
df

,airplane_code,seat_no,fare_conditions
0,32N,1A,Business
1,32N,1C,Business
2,32N,2A,Business
3,32N,2C,Business
4,32N,3A,Business
...,...,...,...
2517,B38M,52C,Economy
2518,B38M,52D,Economy
2519,B38M,52E,Economy
2520,B38M,52F,Economy


In [3]:
print(df.shape)

(2522, 3)


In [4]:
df.head()

,airplane_code,seat_no,fare_conditions
0,32N,1A,Business
1,32N,1C,Business
2,32N,2A,Business
3,32N,2C,Business
4,32N,3A,Business


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2522 entries, 0 to 2521
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   airplane_code    2522 non-null   object
 1   seat_no          2522 non-null   object
 2   fare_conditions  2522 non-null   object
dtypes: object(3)
memory usage: 59.2+ KB


In [7]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.columns

Index(['airplane_code', 'seat_no', 'fare_conditions'], dtype='object')

In [8]:
print(df.isnull().sum())

airplane_code      0
seat_no            0
fare_conditions    0
dtype: int64


In [13]:
for c in df.columns:
    df[c] = df[c].str.strip()

df["airplane_code"] = df["airplane_code"].str.upper()
df["airplane_code"]

0        1A
1        1C
2        2A
3        2C
4        3A
       ... 
2517    52C
2518    52D
2519    52E
2520    52F
2521    53A
Name: seat_no, Length: 2522, dtype: object

In [14]:
df["seat_no"] = df["seat_no"].str.upper()
df["seat_no"]

0        1A
1        1C
2        2A
3        2C
4        3A
       ... 
2517    52C
2518    52D
2519    52E
2520    52F
2521    53A
Name: seat_no, Length: 2522, dtype: object

In [15]:
df["fare_conditions"] = df["fare_conditions"].str.title()
df["fare_conditions"]

0       Business
1       Business
2       Business
3       Business
4       Business
          ...   
2517     Economy
2518     Economy
2519     Economy
2520     Economy
2521     Economy
Name: fare_conditions, Length: 2522, dtype: object

In [18]:
print("Full duplicated rows: ", df.duplicated().sum())
print("Duplicate (airplane, seat): ", df.duplicated(subset = ["airplane_code", "seat_no"]).sum())
df = df.drop_duplicates(subset = ["airplane_code", "seat_no"])

Full duplicated rows:  0
Duplicate (airplane, seat):  0


In [20]:
print(df["seat_no"].str.match(r"^\d{1,3}[A-Z]$").all())

True


In [21]:
print(df["fare_conditions"].value_counts())

fare_conditions
Economy     1666
Comfort      436
Business     420
Name: count, dtype: int64


In [23]:
airplanes = pd.read_csv(r"C:\Users\91766\Downloads\cleaned_data\2_airplanes_clean.csv", dtype = str)
orphans = ~df["airplane_code"].isin(airplanes["airplane_code"])
print("Seats with unknown aircraft: ", orphans.sum())
df = df[~orphans]

Seats with unknown aircraft:  0


In [24]:
check = df.groupby("airplane_code").size().rename("seats_in_map").reset_index()
check = check.merge(airplanes.assign(seat_capacity = airplanes["seat_capacity"].astype(int)),
                    on = "airplane_code")[["airplane_code", "seats_in_map", "seat_capacity"]]
check["difference"] = check["seats_in_map"] - check["seat_capacity"]
check

,airplane_code,seats_in_map,seat_capacity,difference
0,32N,202,166,36
1,339,326,281,45
2,351,377,325,52
3,77W,462,404,58
4,789,302,257,45
5,7M7,196,160,36
6,A21N,259,220,39
7,B38M,213,178,35
8,CR7,89,70,19
9,E70,96,78,18


In [26]:
print(df.shape, 
      df.isnull().sum())

(2522, 3) airplane_code      0
seat_no            0
fare_conditions    0
dtype: int64


In [27]:
import os
out_dir = r"C:\Users\91766\Downloads\cleaned_data"
os.makedirs(out_dir, exist_ok=True)

df.to_csv(os.path.join(out_dir, "3_seats_clean.csv"), index=False)
print("Saved:", len(df), "rows")

Saved: 2522 rows
